In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

1. Загрузите данные close_prices.csv. В этом файле приведены цены
акций 30 компаний на закрытии торгов за каждый день периода.

In [2]:
close_prices = pd.read_csv('close_prices.csv')
close_prices.head()

,date,AXP,BA,CAT,CSCO,CVX,DD,DIS,GE,GS,...,PFE,PG,T,TRV,UNH,UTX,V,VZ,WMT,XOM
0,2013-09-23,76.440002,117.510002,85.029999,24.270000,125.519997,59.409999,64.750000,24.280001,165.250000,...,28.799999,79.279999,34.220001,86.379997,71.820000,109.419998,196.240005,47.980000,76.419998,87.750000
1,2013-09-24,76.070000,119.000000,85.110001,24.139999,124.489998,59.319997,64.320000,24.320000,162.970001,...,28.709999,78.620003,34.090000,85.870003,72.320000,110.000000,193.339996,47.270000,75.750000,87.360001
2,2013-09-25,75.989998,118.510002,84.500000,24.430000,124.070000,59.319997,64.449997,24.230000,162.309998,...,28.490000,77.720001,34.049999,85.980003,71.980003,109.260002,191.559998,46.950001,74.650002,87.139999
3,2013-09-26,76.320000,119.379997,84.199997,23.770000,123.489998,59.509996,65.239998,24.250000,162.289993,...,28.520000,78.050003,34.230000,85.830002,72.160004,109.660004,193.559998,47.669998,74.620003,87.070000
4,2013-09-27,75.889999,118.739998,83.800003,23.330000,122.639999,59.009995,65.190002,24.049999,159.850006,...,28.879999,77.209999,33.980000,85.410004,71.989998,109.360001,193.050003,47.000000,74.360001,86.900002


2. На загруженных данных обучите преобразование PCA с числом
компоненты равным 10. Скольких компонент хватит, чтобы объяс
нить 90% дисперсии?

In [6]:
X = close_prices.iloc[:, 1:].values
pca = PCA(n_components=10)
pca.fit(X)

# Смотрим объясненную дисперсию
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

print("\nОбъясненная дисперсия по компонентам:")
for i, (ev, cum) in enumerate(zip(explained_variance_ratio, cumulative_variance), 1):
    print(f"Компонента {i}: {ev:.4f} (накоплено: {cum:.4f})")

n_components_90 = np.argmax(cumulative_variance >= 0.9) + 1
if cumulative_variance[-1] < 0.9:
    n_components_90 = len(cumulative_variance)
    print(f"\nПредупреждение: 10 компонент объясняют только {cumulative_variance[-1]:.2%} дисперсии")
    print(f"Для 90% нужно больше компонент")

print(f"\nКоличество компонент для объяснения 90% дисперсии: {n_components_90}")


Объясненная дисперсия по компонентам:
Компонента 1: 0.7390 (накоплено: 0.7390)
Компонента 2: 0.1101 (накоплено: 0.8490)
Компонента 3: 0.0500 (накоплено: 0.8990)
Компонента 4: 0.0287 (накоплено: 0.9277)
Компонента 5: 0.0222 (накоплено: 0.9499)
Компонента 6: 0.0193 (накоплено: 0.9692)
Компонента 7: 0.0067 (накоплено: 0.9760)
Компонента 8: 0.0061 (накоплено: 0.9821)
Компонента 9: 0.0032 (накоплено: 0.9853)
Компонента 10: 0.0031 (накоплено: 0.9884)

Количество компонент для объяснения 90% дисперсии: 4


3. Применитепостроенное преобразование к исходным данным и возь
мите значения первой компоненты.

In [13]:
X_pca = pca.transform(X)
feature_names = close_prices.columns[1:].tolist()
first_component = X_pca[:, 0]

print(f"\nПервая компонента (первые 10 значений):")
print(first_component[:10])


Первая компонента (первые 10 значений):
[-50.90240358 -52.84690919 -54.61443917 -52.60056628 -52.3701233
 -54.65341197 -52.81257496 -53.6511457  -56.69272698 -54.40265506]


4. Загрузите информациюобиндексеДоу-Джонсаизфайлаdjia_prices.csv.
Чему равна корреляция Пирсона между первой компонентой и ин
дексом Доу-Джонса?

In [9]:
djia = pd.read_csv('djia_index.csv')
djia.head()

,date,^DJI
0,2013-09-23,15401.379883
1,2013-09-24,15334.589844
2,2013-09-25,15273.259766
3,2013-09-26,15328.299805
4,2013-09-27,15258.240234


In [10]:
djia_values = djia.iloc[:, 1].values
correlation = np.corrcoef(first_component, djia_values)[0, 1]
correlation_rounded = round(correlation, 2)
print(f"\nКорреляция Пирсона между первой компонентой и DJIA: {correlation_rounded:.6f}")


Корреляция Пирсона между первой компонентой и DJIA: 0.910000


5. Какая компания имеет наибольший вес в первой компоненте?

In [16]:
weights = pca.components_[0]
max_idx = np.argmax(weights)
company_max = feature_names[max_idx]
print(f"\nКомпания с наибольшим весом: {company_max} ({weights[max_idx]:.6f})")


Компания с наибольшим весом: V (0.579684)
